# Modelo de skip: resultados

Responsable: Fernando Peña Meade (Ingeniero de ML)

Notebook de presentación (F6). Con lo que se sabe al empezar una canción, el modelo predice si el usuario la va a saltar.

- Lee solo lo que genera el entrenamiento en `ENTREGABLES/04_modelo_ml/` (`metrics.json` y las gráficas): números agregados, ninguna fila, canción ni artista.
- No entrena ni toca el lakehouse: eso lo hace el pipeline (`ml_features`, `ml_train`, `ml_predict` en Dagster).
- Se abre con el kernel del `.venv` (`uv sync --group notebook`). **Antes de un commit: Clear All Outputs.**

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def find_report_dir(start: Path) -> Path:
    """Sube hasta encontrar ENTREGABLES/04_modelo_ml (sirve desde notebooks/ o desde la raiz)."""
    for folder in (start, *start.parents):
        candidate = folder / "ENTREGABLES" / "04_modelo_ml"
        if (candidate / "metrics.json").exists():
            return candidate
    raise FileNotFoundError(
        "No hay ENTREGABLES/04_modelo_ml/metrics.json: corre el pipeline (ml_train) primero."
    )


REPORT = find_report_dir(Path.cwd())
doc = json.loads((REPORT / "metrics.json").read_text(encoding="utf-8"))
print(f"Generado: {doc['generated_at']} | MLflow: version {doc['mlflow']['model_version']}")

## 1. Veredicto: ¿le gana al baseline?

El baseline es la tasa de skip histórica de la canción (si es nueva, la del usuario; si tampoco hay, la de entrenamiento). El modelo tiene que ganarle en **ROC-AUC y PR-AUC del periodo de prueba**, que nunca vio.

In [ ]:
NAMES = {
    "roc_auc": "ROC-AUC",
    "pr_auc": "PR-AUC",
    "f1": "F1",
    "precision": "Precision",
    "recall": "Recall",
    "accuracy": "Exactitud",
    "threshold": "Umbral",
}
m = doc["metrics"]
table = pd.DataFrame(
    {
        f"{who} ({split})": {NAMES[k]: m[who][split][k] for k in NAMES}
        for split in ("validation", "test")
        for who in ("model", "baseline")
    }
).round(3)
verdict = "le gana al baseline" if doc["beats_baseline"] else "NO le gana al baseline"
display(Markdown(f"**El modelo {verdict} en el periodo de prueba.**"))
table

## 2. Split temporal

El corte es por fecha, nunca al azar: se entrena con el pasado y se mide con el futuro, como pasaría en producción. Las reproducciones sin etiqueta no entrenan, pero sí reciben predicción.

In [ ]:
rows = doc["split_rows"]
periods = {
    "train": f"antes del {doc['train_end']}",
    "validation": f"{doc['train_end']} al {doc['validation_end']}",
    "test": f"desde el {doc['validation_end']}",
    "unlabeled": "(sin etiqueta)",
}
rates = {
    "train": doc["train_skip_rate"],
    "validation": m["model"]["validation"]["skip_rate"],
    "test": m["model"]["test"]["skip_rate"],
    "unlabeled": None,
}
pd.DataFrame({"periodo": periods, "filas": rows, "tasa de skip": rates}).round(3)

## 3. Matriz de confusión (prueba)

El umbral se eligió en validación (el que maximiza F1) y se aplica tal cual en prueba.

In [ ]:
t = m["model"]["test"]
pd.DataFrame(
    [[t["tn"], t["fp"]], [t["fn"], t["tp"]]],
    index=["real: no skip", "real: skip"],
    columns=["predicho: no skip", "predicho: skip"],
)

## 4. ¿Qué usa el modelo?

Importancia por permutación en prueba: cuánto baja el ROC-AUC al revolver cada feature. Cerca de 0 (o negativo) = el modelo casi no la usa.

In [ ]:
pd.DataFrame(doc["importance"]).set_index("feature").round(4).head(10)

## 5. Gráficas

In [ ]:
for name in ("roc_pr.png", "confusion_matrix.png", "feature_importance.png"):
    display(Image(filename=str(REPORT / name)))

## 6. Opcional: el registro de MLflow en vivo

Solo si el stack está arriba (`docker compose up -d`). Muestra las versiones de `lakehouse_skip` y cuál tiene el alias `champion` (la que usa la inferencia).

In [ ]:
import os

try:
    from mlflow import MlflowClient

    client = MlflowClient(os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000"))
    versions = client.search_model_versions("name = 'lakehouse_skip'")
    display(
        pd.DataFrame(
            [{"version": v.version, "aliases": ", ".join(v.aliases), "run": v.run_id[:8]} for v in versions]
        ).sort_values("version", key=lambda s: s.astype(int), ascending=False)
    )
except Exception as exc:  # el stack apagado no es un error del notebook
    print(f"MLflow no responde ({type(exc).__name__}); levanta el stack para ver el registro.")